# Fetch L3 Ground Truth from Jira for `full_golden.parquet`

This notebook does only four things:

1. Load `full_golden.parquet`.
2. Extract every unique Epic key from `epic_keys`.
3. Fetch each Epic's configured L3 capabilities from Jira field `customfield_18603`.
4. Save one ground-truth row per Epic/L3 pair to `results/epic_l3_ground_truth_full_golden.xlsx`.

It does **not** run LLM evaluation or invalid-Epic/candidate-coverage checks.

In [ ]:
from pathlib import Path
import ast
import os
import re

import httpx
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display


def find_full_golden_file() -> Path:
    """Find full_golden.parquet in this notebook directory or its parent."""
    filename = "full_golden.parquet"
    for directory in (Path.cwd(), Path.cwd().parent):
        candidate = directory / filename
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not find {filename} from {Path.cwd()}")


PARQUET_PATH = find_full_golden_file()
OUTPUT_PATH = (
    PARQUET_PATH.parent
    / "results"
    / "epic_l3_ground_truth_full_golden.xlsx"
)

print(f"Parquet: {PARQUET_PATH}")
print(f"Output:  {OUTPUT_PATH}")

## Extract unique Epic keys

In [ ]:
def parse_epic_keys(value) -> list[str]:
    """Parse Epic keys from a parquet cell."""
    if value is None:
        return []

    if hasattr(value, "tolist") and not isinstance(value, (str, bytes)):
        value = value.tolist()

    if isinstance(value, (list, tuple, set)):
        return [
            str(item).strip()
            for item in value
            if str(item).strip()
        ]

    try:
        if pd.isna(value):
            return []
    except (TypeError, ValueError):
        pass

    text = str(value).strip()
    if not text:
        return []

    try:
        parsed = ast.literal_eval(text)
    except (SyntaxError, ValueError):
        parsed = None

    if isinstance(parsed, (list, tuple, set)):
        return [
            str(item).strip()
            for item in parsed
            if str(item).strip()
        ]

    return re.findall(r"\b[A-Z][A-Z0-9_]*-\d+\b", text)


df = pd.read_parquet(PARQUET_PATH)

if "epic_keys" not in df.columns:
    raise KeyError("full_golden.parquet does not contain an 'epic_keys' column.")

all_epic_keys = sorted(
    {
        epic_key
        for value in df["epic_keys"]
        for epic_key in parse_epic_keys(value)
    }
)

print(f"Parquet rows: {len(df)}")
print(f"Unique Epics: {len(all_epic_keys)}")
print(all_epic_keys[:20])

## Jira L3 ground-truth fetch

In [ ]:
load_dotenv()

JIRA_BASE_URL = os.environ["JIRA_BASE_URL"].rstrip("/")
JIRA_TOKEN = os.environ["JIRA_TOKEN"]

HEADERS = {
    "Authorization": f"Bearer {JIRA_TOKEN}",
    "Accept": "application/json",
}

L3_CAP_FIELD_ID = "customfield_18603"


def get_epic_l3_cap(epic_key: str) -> dict:
    """Fetch one Epic's configured L3 capabilities from Jira."""
    response = httpx.get(
        f"{JIRA_BASE_URL}/rest/api/2/issue/{epic_key}",
        headers=HEADERS,
        params={"fields": f"summary,{L3_CAP_FIELD_ID}"},
        verify=False,
        timeout=60,
    )
    response.raise_for_status()

    issue = response.json()
    fields = issue.get("fields", {})
    raw_l3 = fields.get(L3_CAP_FIELD_ID) or []

    if not isinstance(raw_l3, list):
        raw_l3 = [raw_l3]

    l3_caps = []
    for item in raw_l3:
        if isinstance(item, dict):
            l3_caps.append(
                {
                    "value": item.get("value"),
                    "id": item.get("id"),
                }
            )
        else:
            l3_caps.append(
                {
                    "value": str(item),
                    "id": None,
                }
            )

    return {
        "epic_key": issue.get("key", epic_key),
        "epic_summary": fields.get("summary"),
        "l3_capabilities": l3_caps,
    }


def split_l3_value(value: str | None) -> tuple[str | None, str | None]:
    """Split 'Capability Name {CAP...}' into name and capability ID."""
    if not value:
        return None, None

    match = re.search(r"\{\s*(CAP\d+)\s*\}\s*$", value)
    if not match:
        return value.strip(), None

    return value[: match.start()].strip(), match.group(1)

## Fetch all Epics and save the GT workbook

In [ ]:
def export_jira_l3_ground_truth(
    epic_keys: list[str],
    output_path: str | Path = OUTPUT_PATH,
) -> pd.DataFrame:
    """Fetch Jira L3 values and export one row per Epic/L3 pair."""
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    rows = []
    unique_epics = sorted(set(epic_keys))

    for index, epic_key in enumerate(unique_epics, start=1):
        print(f"[{index}/{len(unique_epics)}] Fetching {epic_key}")

        try:
            result = get_epic_l3_cap(epic_key)
            l3_caps = result["l3_capabilities"]

            if not l3_caps:
                rows.append(
                    {
                        "epic_key": result["epic_key"],
                        "epic_summary": result["epic_summary"],
                        "l3_capability_id": None,
                        "l3_capability_name": None,
                        "jira_option_id": None,
                        "status": "no_l3_configured",
                        "error": None,
                    }
                )
                continue

            for l3_cap in l3_caps:
                capability_name, capability_id = split_l3_value(
                    l3_cap.get("value")
                )
                rows.append(
                    {
                        "epic_key": result["epic_key"],
                        "epic_summary": result["epic_summary"],
                        "l3_capability_id": capability_id,
                        "l3_capability_name": capability_name,
                        "jira_option_id": l3_cap.get("id"),
                        "status": "ok",
                        "error": None,
                    }
                )

        except Exception as exc:
            rows.append(
                {
                    "epic_key": epic_key,
                    "epic_summary": None,
                    "l3_capability_id": None,
                    "l3_capability_name": None,
                    "jira_option_id": None,
                    "status": "error",
                    "error": str(exc),
                }
            )

    ground_truth = pd.DataFrame(rows)

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        ground_truth.to_excel(
            writer,
            sheet_name="jira_l3_ground_truth",
            index=False,
        )
        worksheet = writer.sheets["jira_l3_ground_truth"]
        worksheet.freeze_panes = "A2"
        worksheet.auto_filter.ref = worksheet.dimensions

        for column_cells in worksheet.columns:
            width = min(
                max(
                    len(str(cell.value or ""))
                    for cell in column_cells
                ) + 2,
                80,
            )
            worksheet.column_dimensions[
                column_cells[0].column_letter
            ].width = width

    print(f"Saved {len(ground_truth)} rows to: {output_path}")
    return ground_truth


gt_l3 = export_jira_l3_ground_truth(all_epic_keys)

## Fetch summary

In [ ]:
status_summary = (
    gt_l3.groupby("status")
    .size()
    .rename("row_count")
    .reset_index()
)

print(f"Unique Epics requested: {len(all_epic_keys)}")
print(f"Unique Epics returned:  {gt_l3['epic_key'].nunique()}")
print(f"Total GT rows:          {len(gt_l3)}")
print(
    "Configured L3 rows:    "
    f"{int((gt_l3['status'] == 'ok').sum())}"
)
print(
    "No L3 configured:      "
    f"{gt_l3.loc[gt_l3['status'] == 'no_l3_configured', 'epic_key'].nunique()}"
)
print(
    "Errors:                "
    f"{gt_l3.loc[gt_l3['status'] == 'error', 'epic_key'].nunique()}"
)

display(status_summary)
display(gt_l3.head(50))